# Usage (editable working copy of `docs/source/usage.rst`)

Edit the Markdown cells (prose/headings) and Python code cells
(examples) below. The Python cells are live -- run them to confirm an
edited example still behaves as commented.

When you're done, **File → Download as → reStructuredText (.rst)**
(or `jupyter nbconvert --to rst usage.ipynb`) gives you a `.rst`
starting point. `nbconvert`'s Markdown→RST conversion is close but
not perfect for Sphinx specifics (cross-references like
`` :doc:`api` ``, the exact `.. code-block:: python` directive
formatting, etc.) -- compare the result against the current
`usage.rst` and touch up those bits by hand before committing.


# Usage

## Constructing values

`Hy` always stores exactly two components, `real` and `imag`,
which are either both plain `Fraction`s (a rank-1, complex-like
value) or both `Hy` instances of the same rank (a higher-rank
value). The constructor normalizes whatever it is given so this
invariant always holds:


In [16]:
# >>> from fractions import Fraction
>>> from hyprat import Hy
>>> from IPython.display import display, Math  # for LaTeX output

>>> z1 = Hy('5/2', '-16/5') # a rational complex number
>>> z2 = Hy(2.5, -3.2)      # same value, from floats
>>> z1 == z2

True

In [17]:
>>> Hy(3)                   # 3 + 0j  (a bare number, still rank 1)

Hy('3', '0')

In [18]:
>>> h1 = Hy(0.5, '2/3')     # mix floats and strings
>>> h2 = Hy('3/4', 0.8)
>>> quat = Hy(h1, h2)       # a rational quaternion
>>> print(f"{quat = }")
>>> print(f"{str(quat) = }")

quat = Hy(Hy('1/2', '2/3'), Hy('3/4', '4/5'))
str(quat) = '(1/2+2/3i+3/4j+4/5k)'


In [31]:
>>> h3 = Hy(Hy(1, 2), Hy(3, 4))
>>> h4 = Hy(Hy(5, 6), Hy(7, 8))
>>> oct = Hy(h3, h4)            # a rational octonion
>>> print(f"{oct = }")
>>> print(f"{str(oct) = }")

oct = Hy(Hy(Hy('1', '2'), Hy('3', '4')), Hy(Hy('5', '6'), Hy('7', '8')))
str(oct) = '(1+2i+3j+4k+5L+6iL+7jL+8kL)'


Three more ways to build a Hy:
* from a flat list of coefficients,
* from a string representation,
* randomly generated

In [34]:
%%html
<style>
table {float: left !important;}
</style>

The dimension of a Hypercomplex number is always a power of 2, referred to here as *rank*, and shown in the following table:

| Hypercomplex | Rank | Dimension |
| ---: | :---: | :---: |
| Rational | 0 | $1 = 2^0$ |
| Complex     | 1 | $2 = 2^1$ |
| Quaternion | 2 | $4 = 2^2$ |
| Octonion | 3 | $8 = 2^3$ |
| in general | n | $d = 2^n$ |

In [32]:
# From a flat list of 2**rank coefficients -- ints, floats,
# Fractions, and fraction strings like '5/2' may be freely mixed:
Hy.from_array([1, '2/3', 3.5, '-1/4'])   # a quaternion

# A random value of a given rank, with a seed for reproducibility:
Hy.random(2, seed=0)                      # a random quaternion


Hy(Hy('3/4', '-8/3'), Hy('7/4', '1'))

## Arithmetic

`+`, `-`, `*` and `/` are all defined recursively via the
Cayley-Dickson construction, so they work uniformly at every rank:


In [21]:
z1, z2 = Hy('1', '2'), Hy('3', '-1')
z1 + z2, z1 - z2, z1 * z2, z1 / z2


(Hy('4', '1'), Hy('-2', '3'), Hy('5', '5'), Hy('1/10', '7/10'))

Multiplication is non-commutative for quaternions and octonions, and
non-associative for octonions, exactly as it should be:


In [22]:
i = Hy(Hy(0, 1), Hy(0, 0))
j = Hy(Hy(0, 0), Hy(1, 0))
i * j   # ->  k
j * i   # -> -k


Hy(Hy('0', '0'), Hy('0', '-1'))

## Parsing and printing

`str()` renders a value the way Python renders `complex` numbers
for rank 1 (using `j`), the customary `a+bi+cj+dk` notation for
rank 2, `i, j, k, L, iL, jL, kL` basis labels for rank 3
("octonions"), and `e1 .. e_{2^rank - 1}` imaginary units for
rank >= 4. `Hy.parse` is the inverse operation:


In [23]:
str(Hy('5/2', '16/5'))          # '(5/2+16/5j)'
Hy.parse('5/2+16/5j')           # == Hy('5/2', '16/5')

str(Hy(Hy(1, 2), Hy(3, 4)))     # '(1+2i+3j+4k)'
Hy.parse('1+2i+3j+4k')          # a rational quaternion

o = Hy(Hy(Hy(1, 2), Hy(3, 4)), Hy(Hy(5, 6), Hy(7, 8)))
str(o)                          # '(1+2i+3j+4k+5L+6iL+7jL+8kL)'
Hy.parse('1+2i+3j+4k+5L+6iL+7jL+8kL') == o   # True


True

`repr()` returns Python source that reconstructs an equal value,
e.g. `Hy('5/2', '-16/5')`.

A value's coefficients can also be read out as a flat list with
`to_array()` -- the inverse of `Hy.from_array()` above -- either as
`Fraction`s, or as strings (handy for JSON or other text-based
serialization):


In [24]:
q = Hy(Hy(1, 2), Hy(3, 4))
q.to_array()               # [Fraction(1), Fraction(2), Fraction(3), Fraction(4)]
q.to_array(as_str=True)    # ['1', '2', '3', '4']
Hy.from_array(q.to_array(as_str=True)) == q   # True


True

## Random values

`Hy.random(rank)` draws a random rank-`rank` value: each of its
`2**rank` coefficients is an independent `Fraction(n, d)`, with
`n` uniform over `[lo, hi]` (default `[-9, 9]`) and `d` uniform
over `[1, dmax]` (default `[1, 6]`). This is handy for examples,
demos, and property-based ("fuzz") testing.

There are a few ways to control reproducibility:


In [25]:
# A one-off seed, scoped to just this call:
Hy.random(2, seed=7)

# Hy.seed(...) fixes a shared default RNG for everything that
# follows, so bare Hy.random(rank) calls become reproducible too:
Hy.seed(2026)
Hy.random(2)


Hy(Hy('-2', '7/5'), Hy('-3', '2'))

In [26]:
# Or bring your own random.Random for full control:
import random
Hy.random(2, rng=random.Random(123))


Hy(Hy('-8/3', '-7/4'), Hy('-1', '-2'))

`rank` must be a positive int (every `Hy` has rank >= 1 by
construction, so there's no rank-0 `Hy`), and `rng`/`seed` are
mutually exclusive.

## Units

`Hy.units(rank)` returns every *unit* element of the rank-`rank`
algebra -- the values with exactly one real coordinate equal to
`+-1` and every other coordinate 0 (`+-1`, `+-i`, `+-j`, ...)
-- as a dict keyed by each unit's string form:


In [27]:
Hy.units(1)
# {'1': Hy('1', '0'), '-1': Hy('-1', '0'), 'j': Hy('0', '1'), '-j': Hy('0', '-1')}

Hy.units(2).keys()
# dict_keys(['1', '-1', 'i', '-i', 'j', '-j', 'k', '-k'])


dict_keys(['1', '-1', 'i', '-i', 'j', '-j', 'k', '-k'])

`some_hy.is_unit()` answers the corresponding membership question
for a single value, without building the whole dict:


In [28]:
Hy(0, 1).is_unit()   # True   (it's j)
Hy(1, 1).is_unit()   # False


False

(`rank == 0` is the one case where the "hypercomplex value" in
question is a plain `Fraction` rather than a `Hy`, so
`Hy.units(0)` returns `{'1': Fraction(1), '-1': Fraction(-1)}`.)

## LaTeX rendering

`some_hy.latex()` renders a value as a LaTeX math expression, which
is especially handy in a Jupyter notebook:


In [29]:
from IPython.display import Math

Math(Hy('5/2', '-16/5').latex())   # displays  5/2 - 16/5 j, with a
                                    # horizontal fraction bar


<IPython.core.display.Math object>

Basis units render the same way [the API reference](api.rst)
describes for `str()` -- `j` at rank 1, `i`/`j`/`k` at rank
2, and `i`/`j`/`k`/`L`/`iL`/`jL`/`kL` at rank 3, all
unchanged -- except that the `e1, e2, ...` labels used from rank 4
up are subscripted (rendered as `e_{1}`, `e_{2}`, ...).

Two keyword-only options are available:

* `vinculum` controls how a non-integer coefficient's fraction bar
  is typeset: `"horizontal"` (the default) uses `\frac{num}{den}`;
  `"diagonal"` uses a plain slash, `num/den`:


In [30]:
Hy('5/2', '-16/5').latex()                     # '\\frac{5}{2}-\\frac{16}{5}j'
Hy('5/2', '-16/5').latex(vinculum='diagonal')   # '5/2-16/5j'


'5/2-16/5j'

* `mode` controls whether the result is wrapped in LaTeX math
  delimiters: `"plain"` (the default) returns the bare expression,
  `"inline"` wraps it in `$...$`, and `"display"` wraps it in
  `\[...\]`.

See [api](api.rst) for the full reference.

---
**RST-specific notes for when you convert this back:**
- The `[the API reference](api.rst)` / `[api](api.rst)` links above
  should become Sphinx cross-references, `` :doc:`the API reference <api>` ``
  and `` :doc:`api` ``, in the final `.rst` -- Markdown links don't
  carry that role automatically.
- Section headers here use `#`/`##`; in the original `.rst` these are
  underlined (`=====`, `-----`) rather than `#`-prefixed.
